# Group

> Merging consecutive diff ops into logical blocks for cleaner display.

In [ ]:
#| default_exp group

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import List, Dict, Any

## Grouping replacements

Raw diffs often produce many small consecutive replace ops.
`group_replacements` merges them into larger logical blocks, including
absorbing small equal connectors (spaces, punctuation) between replaces.

Newlines act as hard boundaries — we never merge across `\n`.

In [ ]:
#| export
def group_replacements(ops: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Merge consecutive replace ops, absorbing small equal connectors between them.
    
    Also merges consecutive same-type simple ops (equal, add, remove).
    Newlines in any op act as merge boundaries."""
    if not ops:
        return []

    merged = []
    i = 0

    while i < len(ops):
        current = ops[i].copy()

        if current["type"] == "replace":
            old_parts = [current["old_text"]]
            new_parts = [current["new_text"]]
            j = i + 1

            current_has_newline = "\n" in current["old_text"] or "\n" in current["new_text"]

            while j < len(ops) and not current_has_newline:
                next_op = ops[j]

                if next_op["type"] == "replace":
                    if "\n" in next_op.get("old_text", "") or "\n" in next_op.get("new_text", ""):
                        break
                    old_parts.append(next_op["old_text"])
                    new_parts.append(next_op["new_text"])
                    j += 1
                elif (next_op["type"] == "equal" and
                      len(next_op.get("text", "")) <= 3 and
                      "\n" not in next_op.get("text", "") and
                      j + 1 < len(ops) and
                      ops[j + 1]["type"] == "replace"):
                    old_parts.append(next_op["text"])
                    new_parts.append(next_op["text"])
                    j += 1
                else:
                    break

            merged.append({
                "type": "replace",
                "old_text": "".join(old_parts),
                "new_text": "".join(new_parts),
            })
            i = j

        elif current["type"] in ("equal", "add", "remove"):
            text_parts = [current.get("text", "")]
            j = i + 1

            while j < len(ops) and ops[j]["type"] == current["type"]:
                text_parts.append(ops[j].get("text", ""))
                j += 1

            merged.append({"type": current["type"], "text": "".join(text_parts)})
            i = j

        else:
            merged.append(current)
            i += 1

    return merged

In [ ]:
# consecutive replaces merge
ops = [
    {"type": "replace", "old_text": "a", "new_text": "x"},
    {"type": "replace", "old_text": "b", "new_text": "y"},
]
result = group_replacements(ops)
assert len(result) == 1
assert result[0] == {"type": "replace", "old_text": "ab", "new_text": "xy"}

In [ ]:
# replaces separated by small equal connector merge
ops = [
    {"type": "replace", "old_text": "a", "new_text": "x"},
    {"type": "equal", "text": " "},
    {"type": "replace", "old_text": "b", "new_text": "y"},
]
result = group_replacements(ops)
assert len(result) == 1
assert result[0] == {"type": "replace", "old_text": "a b", "new_text": "x y"}

In [ ]:
# large equal block prevents merging
ops = [
    {"type": "replace", "old_text": "a", "new_text": "x"},
    {"type": "equal", "text": "long text here"},
    {"type": "replace", "old_text": "b", "new_text": "y"},
]
result = group_replacements(ops)
assert len(result) == 3

In [ ]:
# newlines block merging
ops = [
    {"type": "replace", "old_text": "a", "new_text": "x"},
    {"type": "replace", "old_text": "b\nc", "new_text": "y\nz"},
]
result = group_replacements(ops)
# first replace stays alone, second has newline so doesn't merge
assert len(result) == 2
assert result[0] == {"type": "replace", "old_text": "a", "new_text": "x"}

In [ ]:
# newline in equal connector blocks merging
ops = [
    {"type": "replace", "old_text": "a", "new_text": "x"},
    {"type": "equal", "text": "\n"},
    {"type": "replace", "old_text": "b", "new_text": "y"},
]
result = group_replacements(ops)
assert len(result) == 3

In [ ]:
# consecutive same-type simple ops merge
ops = [
    {"type": "equal", "text": "hello "},
    {"type": "equal", "text": "world"},
]
result = group_replacements(ops)
assert result == [{"type": "equal", "text": "hello world"}]

In [ ]:
# empty
assert group_replacements([]) == []

## Applying semantic groups

After the LLM suggests which ops should be grouped together (as lists of consecutive indices),
`apply_semantic_groups` merges those ops into single blocks.

Mixed-type groups (e.g. equal + replace + add) get combined into a single replace with
reconstructed old/new text.

In [ ]:
#| export
def apply_semantic_groups(ops: List[Dict[str, Any]], groups: List[List[int]]) -> List[Dict[str, Any]]:
    """Apply LLM-suggested semantic grouping to ops.
    
    Each group is a list of consecutive indices to merge into a single op.
    Non-consecutive groups are skipped. Ungrouped ops pass through as-is."""
    if not groups:
        return ops

    result = []
    processed = set()

    for i in range(len(ops)):
        if i in processed:
            continue

        # Find if this index belongs to a group
        group_for_idx = None
        for group in groups:
            if i in group:
                group_for_idx = group
                break

        if group_for_idx:
            group_sorted = sorted(group_for_idx)

            # Must be consecutive
            if group_sorted[-1] - group_sorted[0] + 1 != len(group_sorted):
                result.append(ops[i])
                processed.add(i)
                continue

            group_ops = [ops[idx] for idx in group_sorted]
            types = [op.get("type") for op in group_ops]

            if all(t == "equal" for t in types):
                merged_text = "".join(op.get("text", "") for op in group_ops)
                result.append({"type": "equal", "text": merged_text, "accepted": True})

            elif all(t == "replace" for t in types):
                merged_old = "".join(op.get("old_text", "") for op in group_ops)
                merged_new = "".join(op.get("new_text", "") for op in group_ops)
                result.append({"type": "replace", "old_text": merged_old, "new_text": merged_new, "accepted": False})

            elif all(t == "add" for t in types):
                merged_text = "".join(op.get("text", "") for op in group_ops)
                result.append({"type": "add", "text": merged_text, "accepted": False})

            elif all(t == "remove" for t in types):
                merged_text = "".join(op.get("text", "") for op in group_ops)
                result.append({"type": "remove", "text": merged_text, "accepted": False})

            else:
                # Mixed types -> reconstruct as replace
                old_parts, new_parts = [], []
                for op in group_ops:
                    op_type = op.get("type")
                    if op_type == "equal":
                        old_parts.append(op.get("text", ""))
                        new_parts.append(op.get("text", ""))
                    elif op_type == "replace":
                        old_parts.append(op.get("old_text", ""))
                        new_parts.append(op.get("new_text", ""))
                    elif op_type == "add":
                        new_parts.append(op.get("text", ""))
                    elif op_type == "remove":
                        old_parts.append(op.get("text", ""))

                old_text = "".join(old_parts)
                new_text = "".join(new_parts)

                if old_text and new_text:
                    result.append({"type": "replace", "old_text": old_text, "new_text": new_text, "accepted": False})
                elif new_text:
                    result.append({"type": "add", "text": new_text, "accepted": False})
                elif old_text:
                    result.append({"type": "remove", "text": old_text, "accepted": False})

            processed.update(group_sorted)
        else:
            result.append(ops[i])
            processed.add(i)

    return result

In [ ]:
# no groups -> pass through
ops = [{"type": "equal", "text": "hello"}]
assert apply_semantic_groups(ops, []) == ops

In [ ]:
# group two replaces
ops = [
    {"type": "replace", "old_text": "a", "new_text": "x"},
    {"type": "replace", "old_text": "b", "new_text": "y"},
    {"type": "equal", "text": " end"},
]
result = apply_semantic_groups(ops, [[0, 1]])
assert len(result) == 2
assert result[0] == {"type": "replace", "old_text": "ab", "new_text": "xy", "accepted": False}
assert result[1] == {"type": "equal", "text": " end"}

In [ ]:
# mixed types -> becomes replace
ops = [
    {"type": "remove", "text": "old"},
    {"type": "equal", "text": " "},
    {"type": "add", "text": "new"},
]
result = apply_semantic_groups(ops, [[0, 1, 2]])
assert len(result) == 1
assert result[0] == {"type": "replace", "old_text": "old ", "new_text": " new", "accepted": False}

In [ ]:
# non-consecutive group is skipped (each index handled individually)
ops = [
    {"type": "replace", "old_text": "a", "new_text": "x"},
    {"type": "equal", "text": "middle"},
    {"type": "replace", "old_text": "b", "new_text": "y"},
]
result = apply_semantic_groups(ops, [[0, 2]])  # non-consecutive!
assert len(result) == 3  # nothing merged

In [ ]:
# multiple independent groups
ops = [
    {"type": "add", "text": "a"},
    {"type": "add", "text": "b"},
    {"type": "equal", "text": " mid "},
    {"type": "remove", "text": "c"},
    {"type": "remove", "text": "d"},
]
result = apply_semantic_groups(ops, [[0, 1], [3, 4]])
assert len(result) == 3
assert result[0] == {"type": "add", "text": "ab", "accepted": False}
assert result[1] == {"type": "equal", "text": " mid "}
assert result[2] == {"type": "remove", "text": "cd", "accepted": False}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()